<a href="https://colab.research.google.com/github/Vladikadze/Variational-Sparse-Gaussian-Process-with-a-Nonparametric-Prior/blob/main/stats_proj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import jax
from jax import numpy as jnp
from jax import grad, jit, vmap
from jax import random

First I want to represent a single Gaussian Process on the data

I want to predict NSM based on the date

In [5]:
df = pd.read_csv('/content/Steel_industry_data.csv')
df.head()

,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type
0,01/01/2018 00:15,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load
1,01/01/2018 00:30,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load
2,01/01/2018 00:45,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load
3,01/01/2018 01:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load
4,01/01/2018 01:15,3.82,4.50,0.0,0.0,64.72,100.0,4500,Weekday,Monday,Light_Load


In [6]:
import pandas as pd
import jax.numpy as jnp

# ── 1. Parse dates and sort chronologically ───────────────────────────────────
df['date'] = pd.to_datetime(df['date'], dayfirst=True)
df = df.sort_values('date').reset_index(drop=True)

# ── 2. Stratify by weekday (fit weekday model first) ─────────────────────────
df_weekday = df[df['WeekStatus'] == 'Weekday'].reset_index(drop=True)
df_weekend = df[df['WeekStatus'] == 'Weekend'].reset_index(drop=True)

# ── 3. Temporal train/test split on weekdays ──────────────────────────────────
unique_dates = df_weekday['date'].dt.date.unique()  # sorted, one entry per day
split_date   = unique_dates[int(0.8 * len(unique_dates))]  # 80th percentile date

train_df = df_weekday[df_weekday['date'].dt.date < split_date].reset_index(drop=True)
test_df  = df_weekday[df_weekday['date'].dt.date >= split_date].reset_index(drop=True)

print(f"Train: {len(train_df)} rows | {train_df['date'].dt.date.nunique()} days")
print(f"Test:  {len(test_df)} rows  | {test_df['date'].dt.date.nunique()} days")
print(f"Split date: {split_date}")

# ── 4. Extract X (time of day) and Y (energy) ────────────────────────────────
X_train_raw = train_df['NSM'].values / 86400.0   # normalize to [0, 1]
X_test_raw  = test_df['NSM'].values  / 86400.0

Y_train_raw = train_df['Usage_kWh'].values
Y_test_raw  = test_df['Usage_kWh'].values

# ── 5. Standardize Y using train statistics only ──────────────────────────────
Y_mean = Y_train_raw.mean()
Y_std  = Y_train_raw.std()

X_train = jnp.asarray(X_train_raw)
X_test  = jnp.asarray(X_test_raw)

Y_train = jnp.asarray((Y_train_raw - Y_mean) / Y_std)
Y_test  = jnp.asarray((Y_test_raw  - Y_mean) / Y_std)  # same stats, not recomputed

# ── 6. Keep Load_Type for post-hoc validation only (never used in training) ───
load_type_train = train_df['Load_Type'].values
load_type_test  = test_df['Load_Type'].values
load_type_all   = df_weekday['Load_Type'].values

# ── 7. Full weekday arrays for component assignment validation ─────────────────
X_all = jnp.asarray(df_weekday['NSM'].values / 86400.0)
Y_all = jnp.asarray((df_weekday['Usage_kWh'].values - Y_mean) / Y_std)

# ── 8. Sanity checks ──────────────────────────────────────────────────────────
assert X_train.shape == Y_train.shape
assert X_test.shape  == Y_test.shape
assert float(jnp.min(X_train)) >= 0.0 and float(jnp.max(X_train)) <= 1.0
assert abs(float(jnp.mean(Y_train))) < 0.1   # should be ~0 after standardization
assert abs(float(jnp.std(Y_train))  - 1.0) < 0.1  # should be ~1

print(f"\nX_train: {X_train.shape}, range [{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"Y_train: {Y_train.shape}, mean {Y_train.mean():.3f}, std {Y_train.std():.3f}")
print(f"X_test:  {X_test.shape}")
print(f"Y_test:  {Y_test.shape}")
print(f"\nLoad types in train: {pd.Series(load_type_train).value_counts().to_dict()}")
print(f"Load types in test:  {pd.Series(load_type_test).value_counts().to_dict()}")

Train: 19968 rows | 208 days
Test:  5088 rows  | 53 days
Split date: 2018-10-18

X_train: (19968,), range [0.000, 0.990]
Y_train: (19968,), mean -0.000, std 1.000
X_test:  (5088,)
Y_test:  (5088,)

Load types in train: {'Light_Load': 8936, 'Medium_Load': 6304, 'Maximum_Load': 4728}
Load types in test:  {'Light_Load': 2232, 'Medium_Load': 1632, 'Maximum_Load': 1224}


In [7]:
def rbf_kernel(x1, x2, log_lengthscale, log_signal_var):
    """RBF kernel matrix between x1 (n,) and x2 (m,) → (n, m)"""
    ell = jnp.exp(log_lengthscale)
    sv  = jnp.exp(log_signal_var)
    dist_sq = ((x1[:, None] - x2[None, :]) / ell) ** 2
    return sv * jnp.exp(-0.5 * dist_sq)

In [8]:
M = 50
inducing_points = jnp.linspace(0.0, 1.0, M)  # fixed grid over [0, 1]

def sparse_gp_log_lik(log_lengthscale, log_signal_var, log_noise_var,
                       x, y, z=inducing_points):
    """
    Computes the sparse GP log marginal likelihood.

    Args:
        log_lengthscale : scalar, log of RBF lengthscale
        log_signal_var  : scalar, log of signal variance
        log_noise_var   : scalar, log of noise variance
        x               : (n,) normalized input
        y               : (n,) standardized target
        z               : (M,) inducing point locations

    Returns:
        scalar log marginal likelihood
    """
    n         = x.shape[0]
    noise_var = jnp.exp(log_noise_var)

    # Kernel matrices
    Kmm = rbf_kernel(z, z, log_lengthscale, log_signal_var)          # (M, M)
    Knm = rbf_kernel(x, z, log_lengthscale, log_signal_var)          # (n, M)
    Knn_diag = jnp.exp(log_signal_var) * jnp.ones(n)                 # (n,) diagonal only

    # Numerical stability: jitter on Kmm
    Kmm = Kmm + 1e-6 * jnp.eye(M)

    # Cholesky of Kmm
    Lm = jnp.linalg.cholesky(Kmm)                                    # (M, M)

    # A = Lm⁻¹ Knm^T  →  shape (M, n)
    A = jax.scipy.linalg.solve_triangular(Lm, Knm.T, lower=True)

    # Qnn diagonal = diag(Knm Kmm⁻¹ Knm^T) = sum(A**2, axis=0)
    Qnn_diag = jnp.sum(A ** 2, axis=0)                               # (n,)

    # Trace correction term: tr(Knn - Qnn) / noise_var
    trace_term = jnp.sum(Knn_diag - Qnn_diag) / noise_var

    # B = I + A Aᵀ / noise_var  →  shape (M, M)
    B = jnp.eye(M) + A @ A.T / noise_var
    B = B + 1e-6 * jnp.eye(M)
    Lb = jnp.linalg.cholesky(B)                                      # (M, M)

    # c = Lb⁻¹ A y / noise_var  →  shape (M,)
    Ay = A @ y                                                        # (M,)
    c  = jax.scipy.linalg.solve_triangular(Lb, Ay / noise_var, lower=True)

    # Log marginal likelihood
    log_lik = (
        - 0.5 * n * jnp.log(2 * jnp.pi)
        - jnp.sum(jnp.log(jnp.diag(Lb)))                 # log |B|^(1/2)
        - 0.5 * n * log_noise_var                         # log |σ²I|^(1/2)
        - 0.5 * jnp.dot(y, y) / noise_var                # yᵀy / σ²
        + 0.5 * jnp.dot(c, c)                            # correction for Qnn
        - 0.5 * trace_term                                # tr(Knn - Qnn) / σ²
    )

    return log_lik

In [9]:
def test_single_gp(X_train, Y_train):

    # Initial parameters (in log space)
    params = {
        'log_lengthscale' : jnp.log(jnp.array(0.3)),
        'log_signal_var'  : jnp.log(jnp.array(1.0)),
        'log_noise_var'   : jnp.log(jnp.array(0.1)),
    }

    # Forward pass
    ll = sparse_gp_log_lik(
        params['log_lengthscale'],
        params['log_signal_var'],
        params['log_noise_var'],
        X_train, Y_train
    )
    print(f"Log likelihood: {ll:.4f}")
    assert jnp.isfinite(ll), "Log likelihood is not finite"

    # Gradient check
    grad_fn = jit(grad(sparse_gp_log_lik, argnums=(0, 1, 2)))
    grads = grad_fn(
        params['log_lengthscale'],
        params['log_signal_var'],
        params['log_noise_var'],
        X_train, Y_train
    )
    print(f"Grad log_lengthscale : {grads[0]:.6f}")
    print(f"Grad log_signal_var  : {grads[1]:.6f}")
    print(f"Grad log_noise_var   : {grads[2]:.6f}")
    assert all(jnp.isfinite(g) for g in grads), "Gradient contains non-finite values"

    print("\nStep 1 passed: single sparse GP is finite and differentiable.")


In [10]:
test_single_gp(X_train, Y_train)

Log likelihood: -52190.4922
Grad log_lengthscale : -10049.147461
Grad log_signal_var  : 676.718750
Grad log_noise_var   : 46082.953125

Step 1 passed: single sparse GP is finite and differentiable.


In [11]:
# Extract the date part from the 'date' column
df['day'] = df['date'].dt.date

# Calculate the variance of 'Usage_kWh' for each day
daily_variance_Y = df.groupby('day')['Usage_kWh'].var().reset_index()

# Display the results
print("Variance of Y (Usage_kWh) over days:")
display(daily_variance_Y.head())
print(f"Mean of daily Usage_kWh variance: {daily_variance_Y['Usage_kWh'].mean():.2f}")

Variance of Y (Usage_kWh) over days:


,day,Usage_kWh
0,2018-01-01,0.085524
1,2018-01-02,2146.579963
2,2018-01-03,1458.673208
3,2018-01-04,2724.591862
4,2018-01-05,2502.837954


Mean of daily Usage_kWh variance: 890.69
